# Automobile Insurance Claim Fraud Detection

This notebook builds a cleaner and leakage-resistant machine-learning workflow for classifying automobile insurance claims as fraudulent or non-fraudulent.

The original coursework explored extensive EDA, outlier handling, class balancing, multiple classifiers, cross-validation, and model tuning. This revised version keeps the same project objective while improving the experimental design:

- split the data before fitting preprocessing steps;
- treat `?` as missing data inside the preprocessing pipeline;
- one-hot encode nominal categorical variables instead of assigning arbitrary numeric labels;
- apply scaling only where the model needs it;
- apply SMOTE only inside training folds;
- compare models with stratified cross-validation;
- emphasize fraud-class precision, recall, F1, ROC-AUC and PR-AUC rather than accuracy alone;
- evaluate the selected model once on an untouched test set.

The dataset contains 1,000 insurance claims and uses `fraud_reported` as the binary target.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay, roc_auc_score, average_precision_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [ ]:
# Locate the CSV whether the notebook is run from the repository root,
# a notebooks/ folder, or Google Colab.
candidates = [
    Path("Auto_Insurance_Claims_Data.csv"),
    Path("data/Auto_Insurance_Claims_Data.csv"),
    Path("../data/Auto_Insurance_Claims_Data.csv"),
    Path("/content/Auto_Insurance_Claims_Data.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Auto_Insurance_Claims_Data.csv was not found. "
        "Place it in the current directory or in a data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## Data cleaning and feature engineering

The revision avoids deleting unusual claims merely because they are statistical outliers. In a fraud-detection problem, unusual observations may contain useful signal.

Identifier-like fields are removed, `?` values are converted to missing values, policy coverage limits are split into numeric fields, and date information is converted into interpretable features. Vehicle age is calculated relative to the incident year rather than a hard-coded calendar year.


In [ ]:
df = df.copy()

# Remove the empty export column when present.
df = df.drop(columns=["_c39"], errors="ignore")

# Replace the dataset's missing-value marker.
df = df.replace("?", np.nan)

# Remove high-cardinality identifiers/location strings that are not useful general predictors.
df = df.drop(columns=["policy_number", "incident_location", "insured_zip"], errors="ignore")

# Split policy CSL such as "250/500" into numeric limits.
if "policy_csl" in df.columns:
    csl = df["policy_csl"].str.split("/", expand=True)
    df["person_csl"] = pd.to_numeric(csl[0], errors="coerce")
    df["accident_csl"] = pd.to_numeric(csl[1], errors="coerce")
    df = df.drop(columns=["policy_csl"])

# Parse dates and derive features.
for col in ["policy_bind_date", "incident_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "policy_bind_date" in df.columns:
    df["policy_bind_year"] = df["policy_bind_date"].dt.year
    df["policy_bind_month"] = df["policy_bind_date"].dt.month

if "incident_date" in df.columns:
    df["incident_month"] = df["incident_date"].dt.month
    df["incident_day"] = df["incident_date"].dt.day

if {"incident_date", "auto_year"}.issubset(df.columns):
    df["vehicle_age"] = df["incident_date"].dt.year - df["auto_year"]
    df["vehicle_age"] = df["vehicle_age"].clip(lower=0)
    df = df.drop(columns=["auto_year"])

df = df.drop(columns=["policy_bind_date", "incident_date"], errors="ignore")

print("Cleaned shape:", df.shape)
print(df["fraud_reported"].value_counts(dropna=False))


In [ ]:
# Target distribution
target_counts = df["fraud_reported"].value_counts()
ax = target_counts.plot(kind="bar", title="Fraud Reported – Class Distribution")
ax.set_xlabel("Fraud reported")
ax.set_ylabel("Number of claims")
plt.tight_layout()
plt.show()

print((target_counts / target_counts.sum()).rename("proportion"))


## Train/test split

The test set is separated **before** preprocessing, scaling, encoding, or SMOTE. It remains untouched until the final model evaluation.


In [ ]:
X = df.drop(columns=["fraud_reported"])
y = df["fraud_reported"].map({"N": 0, "Y": 1})

if y.isna().any():
    raise ValueError("Unexpected values were found in fraud_reported.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## Model comparison

SMOTE is placed **inside** each imbalanced-learn pipeline. During cross-validation, synthetic samples are therefore created only from the training portion of each fold.

Three representative classifiers are compared:

- Logistic Regression — interpretable linear baseline;
- Random Forest — nonlinear tree ensemble;
- Extra Trees — randomized tree ensemble.

Because fraud is the minority class, the comparison includes fraud-class F1 and recall as well as ROC-AUC and PR-AUC.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

results = []
pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("smote", SMOTE(random_state=42)),
        ("model", model),
    ])
    pipelines[name] = pipe

    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    row = {"Model": name}
    for metric in scoring:
        row[metric] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    results.append(row)

cv_results = pd.DataFrame(results).sort_values(
    ["pr_auc", "f1"], ascending=False
).reset_index(drop=True)

cv_results[[
    "Model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"
]].round(3)


## Tune the strongest tree ensemble

Extra Trees was the strongest model in the original coursework experiments, so it is retained as the model selected for tuning. The search is performed only on the training data, with preprocessing and SMOTE kept inside the cross-validation pipeline.


In [ ]:
extra_trees_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("smote", SMOTE(random_state=42)),
    ("model", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
])

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    extra_trees_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid.fit(X_train, y_train)

print("Best CV PR-AUC:", round(grid.best_score_, 3))
print("Best parameters:")
grid.best_params_


## Final evaluation on the untouched test set

The tuned pipeline is now evaluated once on the held-out test data. This gives a cleaner estimate of how the model performs on claims that were not used for preprocessing, resampling, cross-validation, or hyperparameter selection.


In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test, y_pred,
    target_names=["Non-fraud", "Fraud"],
    digits=3
))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob), 3))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Non-fraud", "Fraud"],
    values_format="d"
)
plt.title("Final Test Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Final Test ROC Curve")
plt.tight_layout()
plt.show()


In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title("Final Test Precision–Recall Curve")
plt.tight_layout()
plt.show()


## Notes

This notebook intentionally does not reproduce the original project's outlier deletion, global scaling, global Yeo–Johnson transformation, label encoding, or pre-cross-validation resampling. Those operations can remove useful fraud cases, impose artificial order on categories, or leak information from validation/test data.

The revised workflow is intended to make the coursework project more reproducible and suitable for a GitHub portfolio. Results may differ from the original report because the experimental methodology has been corrected.
